# Backbone Experiment: vit_b_32

## Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(""))  # repo root
from config import Config
from data.cifake import get_cifake_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from tqdm.notebook import tqdm
import pandas as pd
import torch

print(f"Device: { 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


/Users/chauc/PycharmProjects/asfr/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## Shared Config

In [2]:
# Only line to change
BACKBONE = "vit_b_32"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.cifake_root    = "../data/raw/cifake"
    cfg.data.num_workers    = 4 # change to 0 if on Windows
    cfg.train.backbone_lr   = 1e-4  
    cfg.train.lr            = 1e-4
    cfg.data.batch_size     = 64
    cfg.data.jpeg_aug       = True
    cfg.backbone.name       = BACKBONE
    cfg.backbone.pretrained = True
    cfg.backbone.frozen     = frozen
    cfg.backbone.img_size   = cfg.data.image_size  
    cfg.fusion.mode         = fusion_mode
    cfg.train.epochs        = 50
    cfg.experiment_name     = f"{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes               = f"CIFAKE · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

# Load data
cfg_data = make_cfg("joint_only") # backbone settings only, fusion mode ignored
train_loader, val_loader, test_loader = get_cifake_loaders(cfg_data)
print(f"Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}")

Train: 85,000  Val: 15,000  Test: 20,000
Train: 85,000  Val: 15,000  Test: 20,000


## Experiment 0: Spatial-Only Baseline
Trains only the spatial backbone as a standalone classifier with no frequency branch.
This gives the honest floor. How well the backbone alone can classify real vs fake.
All delta values in later experiments are relative to this number.


In [5]:
cfg0 = make_cfg("joint_only")  # backbone settings only, fusion mode ignored
cfg0.experiment_name = f"{BACKBONE}_spatial_only"
cfg0.notes           = f"spatial-only baseline, no freq branch, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")
print("All subsequent delta values are relative to this number.")

Device: mps
Training spatial-only baseline (vit_b_32) for 50 epochs...
Train: 85,000  Val: 15,000


Epoch   1/50 | train_loss=0.3175 | val_acc=91.3% | best=0.0% | patience=1/5
  ✓ New best saved (91.3%)


Epoch   2/50 | train_loss=0.2502 | val_acc=91.8% | best=91.3% | patience=0/5
  ✓ New best saved (91.8%)


Epoch   3/50 | train_loss=0.2199 | val_acc=92.0% | best=91.8% | patience=0/5
  ✓ New best saved (92.0%)


Epoch   4/50 | train_loss=0.2017 | val_acc=92.8% | best=92.0% | patience=0/5
  ✓ New best saved (92.8%)


...

Epoch  22/50 | train_loss=0.0592 | val_acc=94.8% | best=94.8% | patience=0/5


Epoch  23/50 | train_loss=0.0568 | val_acc=94.7% | best=94.8% | patience=1/5


Epoch  24/50 | train_loss=0.0535 | val_acc=94.7% | best=94.8% | patience=2/5


Epoch  25/50 | train_loss=0.0486 | val_acc=94.6% | best=94.8% | patience=3/5


Epoch  26/50 | train_loss=0.0439 | val_acc=94.4% | best=94.8% | patience=4/5
  Early stopping at epoch 26 — best val_acc=94.8%

Spatial-only results (vit_b_32):
  Accuracy: 94.6%
  AUC-ROC:  0.988
  F1:       0.946
Results saved → ./results/results.csv  (vit_b_32_spatial_only, acc=0.9464)

Spatial-only floor: 94.6%
All subsequent delta values are relative to this number.


## Experiment 1: Joint-Only Baseline
Both branches concatenated into a single shared classifier. No weighting between branches.
This is the floor — all other experiments are compared against this delta value.


In [5]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)

Running: vit_b_32_joint_only
Device: mps


torch.compile skipped — device is mps

Experiment: vit_b_32_joint_only
Backbone: vit_b_32 | Fusion: joint_only | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5518 | val_acc=93.2% | val_auc=0.980 | val_f1=0.933 | best=0.0% | patience=1/5
  grad norms — freq=0.7935 | spatial=0.6002
  -> Saved best val_acc=93.2%


Epoch   2/50 | train_loss=0.4357 | val_acc=94.2% | val_auc=0.987 | val_f1=0.941 | best=93.2% | patience=0/5
  -> Saved best val_acc=94.2%


Epoch   3/50 | train_loss=0.3939 | val_acc=93.9% | val_auc=0.987 | val_f1=0.941 | best=94.2% | patience=0/5


...

Epoch   9/50 | train_loss=0.2872 | val_acc=95.3% | val_auc=0.990 | val_f1=0.954 | best=95.9% | patience=1/5


Epoch  10/50 | train_loss=0.2742 | val_acc=95.7% | val_auc=0.992 | val_f1=0.957 | best=95.9% | patience=2/5


Epoch  11/50 | train_loss=0.2650 | val_acc=95.9% | val_auc=0.992 | val_f1=0.959 | best=95.9% | patience=3/5
  grad norms — freq=0.8775 | spatial=0.4764


Epoch  12/50 | train_loss=0.2547 | val_acc=96.0% | val_auc=0.992 | val_f1=0.960 | best=95.9% | patience=4/5
  Early stopping at epoch 12 — best val_acc=95.9%

Training complete. Best val accuracy: 95.9%
Results will be logged to CSV after full_evaluation().


0.9591333333333333

In [6]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_32_joint_only.pt

EVALUATION — vit_b_32_joint_only
Backbone: vit_b_32 | Fusion: joint_only | Frozen: False
  Joint accuracy:     95.9%
  AUC-ROC:            0.992
  F1:                 0.959
  Spatial-only:       92.9%
  Freq-only:          89.6%
  Delta (Δ):          +2.9%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (vit_b_32_joint_only, acc=0.9586)


## Experiment 2: Scalar Fusion
Two learned scalars (a, b) weight each branch. Softmax ensures a+b=1 always.
Watch the scalar values logged each epoch. `b`is the frequency weight.
If `b` drops below 0.1 early in training the frequency branch is being ignored.


In [7]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)

Running: vit_b_32_scalar
Device: mps
torch.compile skipped — device is mps

Experiment: vit_b_32_scalar
Backbone: vit_b_32 | Fusion: scalar | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5430 | val_acc=93.8% | val_auc=0.983 | val_f1=0.938 | best=0.0% | patience=1/5
  scalars — spatial=0.492 freq=0.508
  grad norms — freq=0.6695 | spatial=0.7397
  -> Saved best val_acc=93.8%


Epoch   2/50 | train_loss=0.4326 | val_acc=93.9% | val_auc=0.986 | val_f1=0.938 | best=93.8% | patience=0/5
  scalars — spatial=0.491 freq=0.509
  -> Saved best val_acc=93.9%


Epoch   3/50 | train_loss=0.3965 | val_acc=94.5% | val_auc=0.989 | val_f1=0.946 | best=93.9% | patience=0/5
  scalars — spatial=0.490 freq=0.510
  -> Saved best val_acc=94.5%


Epoch   4/50 | train_loss=0.3722 | val_acc=95.3% | val_auc=0.991 | val_f1=0.953 | best=94.5% | patience=0/5
  scalars — spatial=0.489 freq=0.511
  -> Saved best val_acc=95.3%


...

Epoch  20/50 | train_loss=0.1800 | val_acc=96.1% | val_auc=0.994 | val_f1=0.962 | best=96.4% | patience=1/5
  scalars — spatial=0.481 freq=0.519


Epoch  21/50 | train_loss=0.1723 | val_acc=96.4% | val_auc=0.994 | val_f1=0.964 | best=96.4% | patience=2/5
  scalars — spatial=0.481 freq=0.519
  grad norms — freq=0.7996 | spatial=0.5995


Epoch  22/50 | train_loss=0.1649 | val_acc=96.4% | val_auc=0.994 | val_f1=0.964 | best=96.4% | patience=3/5
  scalars — spatial=0.480 freq=0.520


Epoch  23/50 | train_loss=0.1568 | val_acc=96.1% | val_auc=0.994 | val_f1=0.962 | best=96.4% | patience=4/5
  scalars — spatial=0.480 freq=0.520
  Early stopping at epoch 23 — best val_acc=96.4%

Training complete. Best val accuracy: 96.4%
Results will be logged to CSV after full_evaluation().


0.9641333333333333

In [8]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_32_scalar.pt

EVALUATION — vit_b_32_scalar
Backbone: vit_b_32 | Fusion: scalar | Frozen: False
  Joint accuracy:     96.2%
  AUC-ROC:            0.993
  F1:                 0.962
  Spatial-only:       93.7%
  Freq-only:          90.4%
  Delta (Δ):          +2.6%  (freq branch contribution)

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (vit_b_32_scalar, acc=0.9624)


## Experiment 3: Gating Fusion 
A small MLP outputs a per-image gate value in [0,1] controlling how much to trust the frequency branch.
Key metric beyond accuracy: **gate entropy must be > 0.3 nats**.
Below that the gate is outputting near-constant values and is not genuinely adapting per sample.


In [9]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)

Running: vit_b_32_gating
Device: mps
torch.compile skipped — device is mps

Experiment: vit_b_32_gating
Backbone: vit_b_32 | Fusion: gating | Frozen: False
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.5907 | val_acc=93.7% | val_auc=0.985 | val_f1=0.939 | best=0.0% | patience=1/5
  gate — entropy=1.524 nats | mean=0.612 | var=0.0032
  grad norms — freq=0.8900 | spatial=0.4379
  -> Saved best val_acc=93.7%


Epoch   2/50 | train_loss=0.4865 | val_acc=94.0% | val_auc=0.988 | val_f1=0.938 | best=93.7% | patience=0/5
  gate — entropy=1.454 nats | mean=0.598 | var=0.0027
  -> Saved best val_acc=94.0%


Epoch   3/50 | train_loss=0.4412 | val_acc=94.3% | val_auc=0.988 | val_f1=0.944 | best=94.0% | patience=0/5
  gate — entropy=1.322 nats | mean=0.571 | var=0.0020
  -> Saved best val_acc=94.3%


Epoch   4/50 | train_loss=0.4175 | val_acc=94.8% | val_auc=0.989 | val_f1=0.949 | best=94.3% | patience=0/5
  gate — entropy=1.440 nats | mean=0.594 | var=0.0025
  -> Saved best val_acc=94.8%


...

Epoch  15/50 | train_loss=0.2660 | val_acc=95.8% | val_auc=0.993 | val_f1=0.959 | best=96.5% | patience=1/5
  gate — entropy=1.693 nats | mean=0.474 | var=0.0045


Epoch  16/50 | train_loss=0.2506 | val_acc=95.9% | val_auc=0.993 | val_f1=0.959 | best=96.5% | patience=2/5
  gate — entropy=1.668 nats | mean=0.514 | var=0.0042
  grad norms — freq=1.0000 | spatial=0.0074


Epoch  17/50 | train_loss=0.2333 | val_acc=96.4% | val_auc=0.994 | val_f1=0.964 | best=96.5% | patience=3/5
  gate — entropy=1.697 nats | mean=0.504 | var=0.0058


Epoch  18/50 | train_loss=0.2225 | val_acc=96.4% | val_auc=0.994 | val_f1=0.964 | best=96.5% | patience=4/5
  gate — entropy=1.412 nats | mean=0.505 | var=0.0028
  Early stopping at epoch 18 — best val_acc=96.5%

Training complete. Best val accuracy: 96.5%
Results will be logged to CSV after full_evaluation().


0.9646666666666667

In [10]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_32_gating.pt

EVALUATION — vit_b_32_gating
Backbone: vit_b_32 | Fusion: gating | Frozen: False
  Joint accuracy:     96.5%
  AUC-ROC:            0.995
  F1:                 0.965
  Spatial-only:       93.9%
  Freq-only:          91.1%
  Delta (Δ):          +2.6%  (freq branch contribution)

  Gate distribution:
    entropy:  1.191 nats  (OK)
    mean:     0.534
    variance: 0.0022

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (vit_b_32_gating, acc=0.9648)


## Experiment 4: Gating Fusion, Frozen Backbone
Same as Experiment 3 but backbone weights are locked.
Only the projection head, frequency branch, fusion, and joint head train.

If the frequency branch helps more here than in Experiment 3, it means the backbone
was learning to capture spectral information during fine-tuning in Experiment 3, essentially teaching itself what the frequency branch provides.


In [11]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)

Running: vit_b_32_gating_frozen
Device: mps
torch.compile skipped — device is mps

Experiment: vit_b_32_gating_frozen
Backbone: vit_b_32 | Fusion: gating | Frozen: True
Epochs: 50 | LR: 0.0001 | Batch: 64



Epoch   1/50 | train_loss=0.7130 | val_acc=84.3% | val_auc=0.950 | val_f1=0.860 | best=0.0% | patience=1/5
  gate — entropy=2.030 nats | mean=0.564 | var=0.0085
  grad norms — freq=0.9117 | spatial=0.3480
  -> Saved best val_acc=84.3%


Epoch   2/50 | train_loss=0.6021 | val_acc=89.2% | val_auc=0.959 | val_f1=0.894 | best=84.3% | patience=0/5
  gate — entropy=1.907 nats | mean=0.648 | var=0.0066
  -> Saved best val_acc=89.2%


Epoch   3/50 | train_loss=0.5720 | val_acc=89.8% | val_auc=0.963 | val_f1=0.900 | best=89.2% | patience=0/5
  gate — entropy=2.092 nats | mean=0.673 | var=0.0101
  -> Saved best val_acc=89.8%


Epoch   4/50 | train_loss=0.5485 | val_acc=90.0% | val_auc=0.965 | val_f1=0.903 | best=89.8% | patience=0/5
  gate — entropy=2.078 nats | mean=0.679 | var=0.0099
  -> Saved best val_acc=90.0%


...

Epoch  18/50 | train_loss=0.4113 | val_acc=92.6% | val_auc=0.979 | val_f1=0.928 | best=92.9% | patience=1/5
  gate — entropy=2.557 nats | mean=0.672 | var=0.0299


Epoch  19/50 | train_loss=0.4132 | val_acc=92.2% | val_auc=0.979 | val_f1=0.925 | best=92.9% | patience=2/5
  gate — entropy=2.593 nats | mean=0.651 | var=0.0318


Epoch  20/50 | train_loss=0.4055 | val_acc=92.6% | val_auc=0.979 | val_f1=0.925 | best=92.9% | patience=3/5
  gate — entropy=2.559 nats | mean=0.676 | var=0.0296


Epoch  21/50 | train_loss=0.4031 | val_acc=92.8% | val_auc=0.980 | val_f1=0.928 | best=92.9% | patience=4/5
  gate — entropy=2.538 nats | mean=0.684 | var=0.0290
  grad norms — freq=0.9773 | spatial=0.1151
  Early stopping at epoch 21 — best val_acc=92.9%

Training complete. Best val accuracy: 92.9%
Results will be logged to CSV after full_evaluation().


0.9292

In [12]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
)

Loaded checkpoint: ./checkpoints/best_vit_b_32_gating_frozen.pt

EVALUATION — vit_b_32_gating_frozen
Backbone: vit_b_32 | Fusion: gating | Frozen: True
  Joint accuracy:     93.2%
  AUC-ROC:            0.982
  F1:                 0.933
  Spatial-only:       78.6%
  Freq-only:          90.1%
  Delta (Δ):          +14.7%  (freq branch contribution)

  Gate distribution:
    entropy:  2.516 nats  (OK)
    mean:     0.686
    variance: 0.0271

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (vit_b_32_gating_frozen, acc=0.9321)


## Summary: vit_b_32
Compare all four runs. Key columns: accuracy, delta (freq branch contribution), gate_entropy.


In [15]:
df = pd.read_csv("./results/results.csv")
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))

       experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
 vit_b_32_spatial_only joint_only   False    0.9464   0.9879 0.9464        0.0000
   vit_b_32_joint_only joint_only   False    0.9586   0.9923 0.9588        0.0000
       vit_b_32_scalar     scalar   False    0.9624   0.9934 0.9624        0.0000
       vit_b_32_gating     gating   False    0.9648   0.9946 0.9648        1.1906
vit_b_32_gating_frozen     gating    True    0.9321   0.9817 0.9329        2.5160


**What to look for:**
- **Delta** (printed by full_evaluation) - how much the frequency branch added over spatial alone
- **Gate entropy** - must be > 0.3 nats for gating experiments to be valid
- **Frozen vs fine-tuned** - if frozen delta > fine-tuned delta, the backbone was absorbing spectral information during fine-tuning
